In [3]:
from langchain_groq import ChatGroq
from langchain.messages import HumanMessage , SystemMessage , AIMessage
from pydantic import SecretStr
import os

groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
	os.environ["GROQ_API_KEY"] = groq_api_key

In [ ]:
model = ChatGroq(
    model="llama-3.3-70b-versatile", 
    api_key= groq_api_key,
    temperature=0.9,
    timeout=10,
    max_retries=6,
    max_tokens=2048)

In [9]:
conversation = [
    SystemMessage(content="You are a helpful assistant that helps users with their questions."),
    HumanMessage(content="How to train a dragon?"),
    AIMessage(content="As a model I can't train a Dragon"),
    HumanMessage(content="How to train a frog?"),
]

response = model.invoke(conversation)
print(response.content)

Training a frog can be a fun and rewarding experience. While frogs are not typically considered trainable in the same way as dogs or horses, you can still teach them to respond to certain cues and even perform tricks. Here are some steps to help you get started:

1. **Choose a tame frog**: Some frog species are more docile and easier to handle than others. Research and choose a species that is known to be calm and gentle, such as the African Dwarf Frog or the American Green Tree Frog.
2. **Provide a suitable environment**: Create a comfortable and safe enclosure for your frog, with a heat source, humidity, and a hiding place or two. Make sure the enclosure is escape-proof and well-ventilated.
3. **Establish a routine**: Frogs thrive on routine, so establish a regular schedule for feeding, handling, and interacting with your frog.
4. **Start with simple interactions**: Begin by simply observing your frog and letting it get used to your presence. You can start by placing your hand near t

In [11]:
for chunk in model.stream("What color is the sky?"):
    for block in chunk.content_blocks:
        if block["type"] == "reasoning" and (reasoning := block.get("reasoning")):
            print(f"Reasoning: {reasoning}")
        elif block["type"] == "tool_call_chunk":
            print(f"Tool call chunk: {block}")
        elif block["type"] == "text":
            print(block["text"])
        else:
            print(f"Other block type: {block}")

The
 color
 of
 the
 sky
 can
 vary
 depending
 on
 the
 time
 of
 day
 and
 atmospheric
 conditions
.
 Here
 are
 some
 common
 colors
 the
 sky
 can
 appear
:


1
.
 **
Blue
**:
 On
 a
 clear
,
 sunny
 day
,
 the
 sky
 typically
 appears
 blue
 due
 to
 a
 phenomenon
 called
 Ray
leigh
 scattering
,
 where
 shorter
 wavelengths
 of
 light
 (
like
 blue
 and
 violet
)
 are
 scattered
 in
 all
 directions
 by
 the
 tiny
 molecules
 of
 gases
 in
 the
 atmosphere
.

2
.
 **
Red
/
Orange
**:
 During
 sunrise
 and
 sunset
,
 the
 sky
 can
 take
 on
 hues
 of
 red
,
 orange
,
 and
 pink
,
 as
 the
 light
 has
 to
 travel
 through
 more
 of
 the
 atmosphere
,
 scattering
 off
 particles
 and
 molecules
,
 which
 sc
atters
 the
 shorter
 wavelengths
 (
like
 blue
)
 and
 allows
 the
 longer
 wavelengths
 (
like
 red
)
 to
 dominate
.

3
.
 **
Gray
**:
 On
 over
cast
 or
 cloudy
 days
,
 the
 sky
 can
 appear
 gray
,
 as
 the
 light
 is
 scattered
 in
 all
 directions
 by
 the
 water
 dro
ple

In [12]:
full = None
for chunk in model.stream("How to sex? For education purpose only?"):
    full = chunk if full is None else full + chunk

    print(full.content)


For
For educational
For educational purposes
For educational purposes,
For educational purposes, I
For educational purposes, I'll
For educational purposes, I'll provide
For educational purposes, I'll provide a
For educational purposes, I'll provide a general
For educational purposes, I'll provide a general overview
For educational purposes, I'll provide a general overview of
For educational purposes, I'll provide a general overview of the
For educational purposes, I'll provide a general overview of the biological
For educational purposes, I'll provide a general overview of the biological and
For educational purposes, I'll provide a general overview of the biological and anatom
For educational purposes, I'll provide a general overview of the biological and anatomical
For educational purposes, I'll provide a general overview of the biological and anatomical differences
For educational purposes, I'll provide a general overview of the biological and anatomical differences between
For educ

In [25]:
import os
import asyncio
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

@tool
def get_weather(location: str):
    """Returns the weather for a specific location."""
    return f"It is 25°C and sunny in {location}."

tools = [get_weather]

# 3. Create the agent
# create_agent handles the graph runtime automatically
agent = create_agent(model=model, tools=tools)

# 4. Stream events
async def run_agent():
    query = "What is the weather in my city?"
    print(f"User: {query}\n")
    
    # Use astream_events to watch the agent "think" and execute
    async for event in agent.astream_events({"messages": [HumanMessage(content=query)]}, version="v2"):
        kind = event["event"]

        if kind == "on_tool_start":
            print(f"⚙️ [Tool Call] Executing: {event['name']} with {event['data'].get('input')}")
            
        elif kind == "on_tool_end":
            print(f"✅ [Tool Output] {event['data'].get('output')}")
            
        elif kind == "on_chat_model_stream":
            content = event["data"]["chunk"].content
            if content:
                print(content, end="", flush=True)

output = await run_agent()
print("\n\nAgent run complete.")
print(f"Final output: {output}")

User: What is the weather in my city?

⚙️ [Tool Call] Executing: get_weather with {'location': "user's city"}
✅ [Tool Output] content="It is 25°C and sunny in user's city." name='get_weather' tool_call_id='sp58nrpaz'
I don't have the ability to know your city. To get the weather for your city, I would need to know the name of your city. If you provide the name of your city, I can use the 'get_weather' function to get the current weather for that location.

Agent run complete.
Final output: None


In [26]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

content="Parrots have colorful feathers for several reasons, including:\n\n1. **Mating and Courtship**: Bright colors can attract a mate and signal a parrot's health, strength, and genetic quality. During courtship, male parrots will often display their vibrant plumage to impress potential mates.\n2. **Communication and Social Status**: Colorful feathers can help parrots communicate with each other, conveying information about their identity, social status, and intentions. For example, some parrot species have distinctive colors that signal dominance or submission.\n3. **Camouflage and Defense**: While it might seem counterintuitive, some parrots' bright colors can actually serve as camouflage in their natural environments. For instance, the blue and yellow feathers of the blue-and-yellow macaw can help it blend in with the bright flowers and fruits of the tropical rainforest.\n4. **Thermoregulation**: In some cases, the color of a parrot's feathers can help regulate its body temperatu

In [31]:
for response in model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(list(response)[1].content)

Airplanes fly by using a combination of four main forces: lift, weight, thrust, and drag. Here's a simplified explanation of how these forces work together to make an airplane fly:

1. **Lift**: Lift is the upward force that opposes the weight of the plane and keeps it flying. It's created by the shape of the wings, which are curved on top and flat on the bottom. As the plane moves forward, the air flows over and under the wing, creating an area of lower air pressure above the wing and an area of higher air pressure below it. This pressure difference creates an upward force, or lift, that counteracts the weight of the plane.
2. **Weight**: Weight is the downward force that pulls the plane towards the ground. It's the combined weight of the plane itself, the passengers, cargo, and fuel.
3. **Thrust**: Thrust is the forward force that propels the plane through the air. It's created by the plane's engines, which produce a stream of high-speed air that exits the back of the plane. As this 

In [56]:
# Making a Tool
messages: list = [SystemMessage(content="You are a helpful assistant that helps users with their questions."),
                  HumanMessage(content="What is the weather like today in New York?")]

@tool
def get_weather(location: str):
    """Returns the current Weather."""
    return f"It is 25°C and sunny in {location}."

model_with_tools = model.bind_tools([get_weather])
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)
# print(ai_message.tool_calls)

for tool_call in ai_message.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    # print(f"Tool result: {tool_result}")
    messages.append(tool_result)

# print(messages)
final_response = model_with_tools.invoke(messages)
print(final_response.content)

The current weather in New York is 25°C and sunny.


In [62]:
## More than 1 tool call in a single response
messages: list = [SystemMessage(content="You are a helpful assistant that helps users with their questions."),
                  HumanMessage(content="What is the weather like today in New York and Mumbai? and what is the time in both cities?")]

@tool
def get_weather(location: str):
    """Returns the current Weather."""
    return f"It is 25°C and sunny in {location}."

@tool
def get_time(location: str):
    """Returns the current time."""
    return f"The current time in {location} is 2:00 PM."

model_with_tools = model.bind_tools([get_weather, get_time])
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_result = get_weather.invoke(tool_call)
    elif tool_call["name"] == "get_time":
        tool_result = get_time.invoke(tool_call)
    messages.append(tool_result)

# print(messages)
final_response = model_with_tools.invoke(messages)
print(final_response.content)

The weather in New York is 25°C and sunny, and in Mumbai it is also 25°C and sunny. The current time in New York is 2:00 PM, and in Mumbai it is 11:30 PM.


In [74]:
## Structured Output
from pydantic import BaseModel

class Cast(BaseModel):
    name: str
    role: str

class Movie(BaseModel):
    title: str
    director: str
    year: int
    intimate_scene: bool
    cast: list[Cast]

model_with_structured_output = model.with_structured_output(Movie)
response = model_with_structured_output.invoke([
    SystemMessage(content="You are a movie database assistant. You provide information about movies in a structured format."),
    HumanMessage(content="Tell me about the movie Michael 2026.")])
print(response)

title='Michael' director='' year=2026 intimate_scene=False cast=[]


In [84]:
## Server side tool use
tool1 = {"type": "web_search"}
model_with_server_tool = model.bind_tools([tool1])

response = model_with_server_tool.invoke(["What is the current date and time in India?"])
print(response)

BadRequestError: Error code: 400 - {'error': {'message': 'code=400, message=tools[0].type must be one of [function, mcp], type=invalid_request_error', 'type': 'invalid_request_error'}}

In [83]:
from langchain_core.callbacks import UsageMetadataCallbackHandler
callback = UsageMetadataCallbackHandler()

result = model.invoke("Hello", config={"callbacks": [callback]})

print(callback.usage_metadata)

{'llama-3.3-70b-versatile': {'input_tokens': 36, 'output_tokens': 10, 'total_tokens': 46}}


In [ ]:
## Tool calling custom name and description

@tool("web_search" , description="Search the web for information.")  # Custom name
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"

print(search.invoke("What is the capital of France?"))

TypeError: 'dict' object is not callable

In [ ]:
# Advanced Schema
from pydantic import BaseModel, Field
from typing import Literal

class WeatherSchema(BaseModel):
    """Input Schema for Weather Queries"""
    location: str = Field(description="This is the location of the place that need a weather check")
    units: Literal["celcius", "fahrenheit"] = Field(
        default="celcius",
        description="Temperature Unit"
    )
    include_forcast: bool = Field(
        default= False,
        description="Include 5 days forcast"
    )

@tool(args_schema=WeatherSchema)
def get_weather(
    location: str,
    units: str = "celcius" , 
    include_forcast: bool = False) -> str:
    
    """Get the current weather of the location"""
    return ""

In [8]:
## Access the State
from langchain.tools import tool , ToolRuntime
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Annotated
from langgraph.prebuilt import InjectedState

@tool
def get_last_user_message(state: Annotated[dict, InjectedState]):
    """Get the most recent message of the user"""
    messages = state.get("messages", [])
    
    # Finding the last human message
    for message in reversed(messages):
        if isinstance(message , HumanMessage):
            return message.content

    return "No last message found"

agent = create_agent(
    model=model,
    tools=[get_last_user_message],
    system_prompt="you are a helping agent"
)

#  THIS WILL WORK
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Hello! I love coding in Python."},
        {"role": "user", "content": "What is the most recent Human message?"}
    ]
})

print(result["messages"][-1].content)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmjhh9rtfvn83napwt12tfaf` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99739, Requested 654. Please try again in 5m39.552s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}